# ¿Ingreso y precio de la vivienda: asociación o confounding espacial? (California, 1990)
## Intersección multivariable + causalidad observacional — evidencia condicional, no identificación causal definitiva

**Pregunta central:** En el dataset California Housing (censo 1990, *block groups*), ¿cómo cambia la asociación entre ingreso mediano (`MedInc`) y valor mediano de la vivienda (`MedHouseVal`) al condicionar por características de la vivienda y ubicación geográfica, y qué indica esto sobre posible confounding espacial y heterogeneidad?

> **Alcance inferencial:** se reportan asociaciones condicionales y resultados predictivos/diagnósticos según los modelos efectivamente ejecutados en el entorno. En datos observacionales transversales sin estrategia de identificación cuasi-experimental, esto **no** equivale a estimar un efecto causal definitivo.

---

## 1. Contexto: el debate vivienda-ingreso-ubicación (½ cuartilla)

California es un caso emblemático de tensiones entre ingreso, oferta, amenities y localización. El dataset `california_housing.csv` (derivado de Pace & Barry, 1997; disponible vía scikit-learn) contiene ~20 mil *block groups* del censo de 1990 con variables socioeconómicas, de vivienda y coordenadas: `MedInc`, `HouseAge`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude`, `Longitude` y `MedHouseVal`.

La correlación bruta entre `MedInc` y `MedHouseVal` es positiva, pero no distingue cuánto se debe a ingreso y cuánto a ubicación/amenities correlacionadas. La historia analítica consiste en **condicionar progresivamente** (controles observables, estructura espacial cuando está disponible, modelos flexibles y diagnósticos) para evaluar la estabilidad de la asociación y su heterogeneidad.

**Hipótesis (formulación condicional/observacional):**
- H₀: una vez condicionando por covariables observables y ubicación, `MedInc` no aporta asociación adicional relevante con `MedHouseVal`.
- H₁: `MedInc` mantiene una asociación condicional positiva (potencialmente heterogénea en el espacio), sin interpretar esto como causalidad identificada.

**Parámetros/diagnósticos de interés (según disponibilidad):** coeficientes OLS (simple y con controles), diagnóstico de autocorrelación espacial en residuos (Moran's I si hay dependencias), comparación con modelos espaciales/ML si están disponibles, heterogeneidad descriptiva/predictiva y sensibilidad a confounders no observados (Oster) como ejercicio exploratorio.

## 2. Dataset: 20,640 block groups, censo 1990 (½ cuartilla)

In [ ]:
# Carga
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv(Path("../data/raw/california_housing.csv"))
print(f"Shape: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")
print(df.describe())
print(f"\nCorr MedInc-MedHouseVal: {df['MedInc'].corr(df['MedHouseVal']):.4f}")
print(f"Corr MedInc-Latitude: {df['MedInc'].corr(df['Latitude']):.4f}")
print(f"Corr MedHouseVal-Latitude: {df['MedHouseVal'].corr(df['Latitude']):.4f}")

In [ ]:
import sys
from pathlib import Path
_ROOT = Path('..').resolve()
for _p in [str(_ROOT), str(_ROOT / 'src')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Top-coding en target (5.0 = $500k)
print(f"MedHouseVal == 5.0: {(df['MedHouseVal'] == 5.0).sum()} ({(df['MedHouseVal'] == 5.0).mean():.2%})")
print(f"MedHouseVal > 4.5: {(df['MedHouseVal'] > 4.5).sum()} ({(df['MedHouseVal'] > 4.5).mean():.2%})")

# Distribución espacial
import matplotlib.pyplot as plt
from src.viz import setup_plotting_style, save_fig
sys.path.append("../src")
setup_plotting_style()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sc = axes[0].scatter(df.Longitude, df.Latitude, c=df.MedHouseVal, cmap='viridis', s=1, alpha=0.6)
axes[0].set_title('MedHouseVal (espacial)')
plt.colorbar(sc, ax=axes[0])

sc = axes[1].scatter(df.Longitude, df.Latitude, c=df.MedInc, cmap='plasma', s=1, alpha=0.6)
axes[1].set_title('MedInc (espacial)')
plt.colorbar(sc, ax=axes[1])

sc = axes[2].scatter(df.Longitude, df.Latitude, c=df.AveOccup, cmap='inferno', s=1, alpha=0.6)
axes[2].set_title('AveOccup (espacial)')
plt.colorbar(sc, ax=axes[2])

plt.tight_layout()
save_fig("05_spatial_dist")
plt.show()

**Fuente:** `sklearn.datasets.fetch_california_housing` (Pace & Barry 1997, *Sparse Spatial Autoregressions*).
**Tamaño:** 20,640 block groups (censo 1990).
**Variables:**
- `MedInc`: Ingreso mediano (decenas de miles USD)
- `HouseAge`: Edad media vivienda
- `AveRooms`: Habitaciones medias por hogar
- `AveBedrms`: Dormitorios medios por hogar
- `Population`: Población block group
- `AveOccup`: Ocupación media (personas/hogar)
- `Latitude`, `Longitude`: Centroide block group
- `MedHouseVal`: Valor mediano vivienda (cientos de miles USD) — **TARGET**

**Sesgos y limitaciones críticas — léanlos antes de seguir:**
- **Datos 1990 (34 años viejo):** Relaciones pueden no sostener post-2008 / post-COVID / work-from-home. Es un mercado inmobiliario que ya no existe igual.
- **Top-coding en target:** 5.0 = $500k+ (censoring en cola alta: 5.2% observaciones). Silicon Valley está censurado.
- **MAUP (Modifiable Areal Unit Problem):** Block groups son unidades administrativas arbitrarias; resultados sensibles a agregación.
- **Confounding no observado:** Amenities (playas, vistas, parques), zoning restrictivo, transporte público, calidad escuelas, crimen.
- **Autocorrelación espacial:** Violación independencia OLS; errores estándar sesgados.

## 3. El espacio importa: mapa, PDP y residuos (¾ cuartilla)

In [ ]:
# 1. Scatter MedInc vs MedHouseVal coloreado por lat/long (costa=rojo)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc = axes[0].scatter(df.MedInc, df.MedHouseVal, c=df.Latitude, cmap='RdBu_r', s=3, alpha=0.5)
axes[0].set_xlabel('MedInc ($10k)')
axes[0].set_ylabel('MedHouseVal ($100k)')
axes[0].set_title('MedInc vs MedHouseVal (color=Latitud)')
plt.colorbar(sc, ax=axes[0], label='Latitud')

# Añadir regresión lineal simple
from scipy.stats import linregress
slope, intercept, r, p, se = linregress(df.MedInc, df.MedHouseVal)
x_line = np.linspace(df.MedInc.min(), df.MedInc.max(), 100)
axes[0].plot(x_line, intercept + slope*x_line, 'k--', label=f'OLS: β={slope:.3f}, R²={r**2:.3f}')
axes[0].legend()

# 2. Partial Dependence Plot (PDP) MedInc -> MedHouseVal
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import partial_dependence

X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

gbr = HistGradientBoostingRegressor(random_state=42, max_iter=200)
gbr.fit(X, y)

pdp_result = partial_dependence(gbr, X, ['MedInc'], percentiles=(0.01, 0.99), grid_resolution=30)
grid = pdp_result.get('grid_values', pdp_result.get('values'))
pdp_vals = np.asarray(grid[0] if isinstance(grid, (list, tuple)) else grid).ravel()
pdp_avg = np.asarray(pdp_result['average'])
pdp_avg = pdp_avg[0] if pdp_avg.ndim > 1 else pdp_avg
pdp_avg = pdp_avg.ravel()

axes[1].plot(pdp_vals, pdp_avg, 'r-', linewidth=3, label='PDP (GBT)')
axes[1].set_xlabel('MedInc ($10k)')
axes[1].set_ylabel('E[MedHouseVal | MedInc] ($100k)')
axes[1].set_title('Partial Dependence: MedInc → MedHouseVal')
axes[1].axhline(y.mean(), color='gray', linestyle='--', label='Media marginal')
axes[1].legend()

plt.tight_layout()
save_fig("05_pdp_medinc")
plt.show()

print(f"OLS simple: MedHouseVal = {intercept:.3f} + {slope:.3f}×MedInc (R²={r**2:.3f})")
print(f"PDP en MedInc=3: {np.interp(3, pdp_vals, pdp_avg):.3f}, en MedInc=8: {np.interp(8, pdp_vals, pdp_avg):.3f}")
print(f"Efecto marginal PDP (3→8): {np.interp(8, pdp_vals, pdp_avg) - np.interp(3, pdp_vals, pdp_avg):.3f} $100k = ${(np.interp(8, pdp_vals, pdp_avg) - np.interp(3, pdp_vals, pdp_avg))*100:.0f}k")


In [ ]:
# 3. Residuales espaciales: OLS con controles + Moran's I (si disponible)
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np

# OLS baseline con controles
ols = smf.ols('MedHouseVal ~ MedInc + HouseAge + AveRooms + AveBedrms + Population + AveOccup + Latitude + Longitude', df).fit()
print(ols.summary().tables[1])

df['resid_ols'] = ols.resid

# Moran's I en residuos (requiere libpysal/esda)
from stats_helpers import morans_i
coords = df[['Latitude', 'Longitude']].values
mi = morans_i(df['resid_ols'].values, coords, k=5)
moran_available = isinstance(mi, dict) and ('I' in mi) and np.isfinite(mi.get('I', np.nan))
if moran_available:
    print(f"\nMoran's I (residuos OLS): I={mi.get('I', np.nan):.4f}, p_sim={mi.get('p_sim', np.nan):.4f}, p_norm={mi.get('p_norm', np.nan):.4f}")
else:
    print(f"\nMoran's I no disponible en este entorno o no calculable: {mi}")

# Mapa residuos
import matplotlib.pyplot as plt
from src.viz import save_fig
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(df.Longitude, df.Latitude, c=df['resid_ols'], cmap='RdBu', s=5, alpha=0.7, vmin=-2, vmax=2)
ax.set_title('Residuos OLS (rojo=subestima, azul=sobrestima)')
plt.colorbar(sc, ax=ax, label='Residuo')
save_fig("05_residuals_spatial")
plt.show()


In [ ]:
# 4. Heterogeneidad: PDP condicionado a cuartiles de latitud
df['lat_q'] = pd.qcut(df['Latitude'], 4, labels=['Q1 (Sur)', 'Q2', 'Q3', 'Q4 (Norte)'])

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
axes = axes.ravel()

for i, q in enumerate(['Q1 (Sur)', 'Q2', 'Q3', 'Q4 (Norte)']):
    subset = df[df['lat_q'] == q]
    if len(subset) > 100:
        X_sub = subset.drop(['MedHouseVal', 'lat_q'], axis=1)
        y_sub = subset['MedHouseVal']
        gbr_sub = HistGradientBoostingRegressor(random_state=42, max_iter=100)
        gbr_sub.fit(X_sub, y_sub)

        pdp = partial_dependence(gbr_sub, X_sub, ['MedInc'], percentiles=(0.05, 0.95), grid_resolution=20)
        grid = pdp.get('grid_values', pdp.get('values'))
        vals = np.asarray(grid[0] if isinstance(grid, (list, tuple)) else grid).ravel()
        avg = np.asarray(pdp['average'])
        avg = avg[0] if avg.ndim > 1 else avg
        avg = avg.ravel()
        axes[i].plot(vals, avg, 'r-', linewidth=2)
        axes[i].set_title(f'{q} (n={len(subset)})')
        axes[i].set_xlabel('MedInc')
        axes[i].set_ylabel('E[MedHouseVal|MedInc]')
        axes[i].axhline(y_sub.mean(), color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
save_fig("05_pdp_by_lat")
plt.show()


In [ ]:
# 5. Feature importance: permutation + SHAP
from sklearn.inspection import permutation_importance

# Permutation importance
perm = permutation_importance(gbr, X, y, n_repeats=10, random_state=42, n_jobs=-1)
perm_df = pd.DataFrame({
    'feature': X.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(perm_df['feature'], perm_df['importance_mean'], xerr=perm_df['importance_std'])
ax.set_xlabel('Permutation Importance (drop en R²)')
ax.set_title('Feature Importance (HistGradientBoosting)')
plt.tight_layout()
save_fig("05_perm_importance")
plt.show()

print(perm_df.to_string(index=False))

> **Lectura basada en evidencia observada:** los mapas y las distribuciones espaciales sugieren que ubicación y submercados locales importan. Al comparar asociaciones brutas y condicionales, y al inspeccionar residuos/heterogeneidad, se evalúa si `MedInc` retiene señal una vez considerados controles y geografía. Las conclusiones deben leerse como evidencia asociacional/predictiva condicionada al dataset y a los modelos efectivamente ejecutados.

**Hallazgos a interpretar desde las salidas:**
1. **Patrón espacial:** concentración geográfica de valores e ingresos.
2. **Cambio al condicionar:** diferencia entre coeficiente simple y especificaciones con controles/ubicación.
3. **Diagnósticos espaciales:** solo se interpretan si fueron calculados (p. ej. Moran's I disponible).
4. **Heterogeneidad:** variación de respuestas parciales/PDP entre zonas, como señal descriptiva de no uniformidad.

In [ ]:
# Registrar hallazgos (dinámicos y condicionados a lo efectivamente calculado)
from narrative import hallazgo
from IPython.display import display, Markdown
import numpy as np

# OLS simple (slope, se, p calculados en celda de PDP/scatter)
display(hallazgo(
    f"Asociación bruta: correlación MedInc–MedHouseVal = {df['MedInc'].corr(df['MedHouseVal']):.3f}; OLS bivariado β_MedInc = {slope:.3f} (unidades: $100k por $10k de ingreso), IC95% aprox. [{slope-1.96*se:.3f}, {slope+1.96*se:.3f}], p={p:.3g}.",
    evidencia="OLS simple (scatter/PDP) sobre el dataset cargado",
    ic=(float(slope-1.96*se), float(slope+1.96*se)),
    pval=float(p) if np.isfinite(p) else None
))

# OLS con controles (si existe)
try:
    b_ctrl = float(ols.params['MedInc'])
    ci_ctrl = ols.conf_int().loc['MedInc'].values
    display(hallazgo(
        f"Asociación condicional (OLS con controles y coordenadas): β_MedInc = {b_ctrl:.3f}, IC95% [{float(ci_ctrl[0]):.3f}, {float(ci_ctrl[1]):.3f}]. Comparar con la estimación bruta permite evaluar cuánto de la asociación se atenúa al condicionar por covariables observables y ubicación (interpretación asociacional, no causal definitiva).",
        evidencia="OLS múltiple: MedHouseVal ~ MedInc + HouseAge + AveRooms + AveBedrms + Population + AveOccup + Latitude + Longitude",
        ic=(float(ci_ctrl[0]), float(ci_ctrl[1]))
    ))
except Exception:
    pass

# Moran's I condicional
if 'moran_available' in globals() and moran_available:
    mi_I = float(mi.get('I', np.nan)); mi_p = float(mi.get('p_sim', mi.get('p_norm', np.nan)))
    display(hallazgo(
        f"Diagnóstico espacial de residuos OLS: Moran's I = {mi_I:.4f} (p={mi_p:.4g}), lo que sugiere dependencia espacial residual y la conveniencia de tratar la inferencia/errores con cautela.",
        evidencia="Moran's I (k=5 vecinos) sobre residuos del OLS con controles",
        ic=None,
        pval=float(mi_p) if np.isfinite(mi_p) else None
    ))
else:
    display(Markdown("**Hallazgo (diagnóstico espacial):** Moran's I no se calculó en esta ejecución (dependencias espaciales no disponibles o resultado no finito). No se afirma un valor numérico de autocorrelación; se mantiene la advertencia metodológica sobre posible dependencia espacial en datos georreferenciados."))

display(hallazgo(
    "Las exploraciones de dependencia parcial por subzonas (p. ej. cuartiles de latitud) sugieren heterogeneidad espacial en la relación condicional entre ingreso y valor de vivienda dentro de este dataset.",
    evidencia="PDP global y PDP condicionados por latitud (modelos de árboles/boosting)",
    ic=None
))


## 4. Escalera causal: OLS → SAR → GBT + SHAP → Causal Forest → Oster (1 cuartilla)

In [ ]:
# 1. Modelo espacial (SAR) si está disponible
sar = None
sar_beta_medinc = np.nan
sar_rho = np.nan
sar_p = np.nan
try:
    from spreg import ML_Lag
    from libpysal.weights import KNN
    
    y_sar = df['MedHouseVal'].values.reshape(-1, 1)
    X_sar = df[['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']].values
    var_names = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
    
    w = KNN.from_array(df[['Latitude', 'Longitude']].values, k=5)
    w.transform = 'r'
    
    sar = ML_Lag(y_sar, X_sar, w=w, name_y='MedHouseVal', name_x=var_names)
    sar_beta_medinc = float(np.asarray(sar.betas[0]).ravel()[0])
    sar_rho = float(np.asarray(sar.rho).ravel()[0])
    sar_p = float(np.asarray(sar.z_stat[0]).ravel()[1]) if hasattr(sar, 'z_stat') else np.nan
    print("=== SPATIAL LAG (SAR) ===")
    print(sar.summary)
    print(f"\nRho (autocorrelación espacial): {sar_rho:.4f}")
    print(f"β_MedInc (SAR): {sar_beta_medinc:.4f}")
except Exception as e:
    print(f"SAR no disponible en este entorno: {e}")
    sar = None


In [ ]:
# 2. Gradient Boosting + SHAP
try:
    import shap
    
    explainer = shap.TreeExplainer(gbr)
    shap_values = explainer.shap_values(X)
    
    # SHAP summary
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_values, X, plot_type='bar', show=False)
    plt.title('SHAP Feature Importance (mean |SHAP|)')
    save_fig("05_shap_bar")
    plt.show()
    
    # SHAP dependence MedInc
    fig, ax = plt.subplots(figsize=(8, 5))
    shap.dependence_plot('MedInc', shap_values, X, interaction_index='Latitude', show=False)
    plt.title('SHAP Dependence: MedInc coloreado por Latitud')
    save_fig("05_shap_dependence")
    plt.show()
except ImportError:
    print("SHAP no instalado - saltando")

In [ ]:
# 3. Causal Forest (econml) para heterogeneidad condicional (si disponible)
cate = None
cate_mean = np.nan
cate_std = np.nan
cate_p5 = np.nan
cate_p95 = np.nan
try:
    from econml.dml import CausalForestDML
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import HistGradientBoostingRegressor
    
    T = df['MedInc'].values.reshape(-1, 1)
    Y = df['MedHouseVal'].values
    X_cf = df.drop(['MedHouseVal', 'MedInc'], axis=1).values
    
    X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
        X_cf, T, Y, test_size=0.2, random_state=42
    )
    
    cf = CausalForestDML(
        model_y=HistGradientBoostingRegressor(random_state=42),
        model_t=HistGradientBoostingRegressor(random_state=42),
        n_estimators=200,
        min_samples_leaf=10,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    cf.fit(Y_train, T_train, X=X_train)
    cate = cf.effect(X_test)
    cate_mean = float(np.mean(cate)); cate_std = float(np.std(cate))
    cate_p5 = float(np.percentile(cate, 5)); cate_p95 = float(np.percentile(cate, 95))
    print(f"CATE media (estimación condicional): {cate_mean:.4f}")
    print(f"CATE std: {cate_std:.4f}")
    print(f"CATE 5%-95%: [{cate_p5:.4f}, {cate_p95:.4f}]")
    
    lat_test = X_test[:, 5]
    import matplotlib.pyplot as plt
    from src.viz import save_fig
    fig, ax = plt.subplots(figsize=(8, 5))
    sc = ax.scatter(lat_test, cate, c=cate, cmap='RdBu', alpha=0.6, s=10)
    ax.set_xlabel('Latitud')
    ax.set_ylabel('Efecto condicional estimado')
    ax.set_title('Heterogeneidad condicional (Causal Forest, si disponible)')
    plt.colorbar(sc, ax=ax, label='Estimación')
    save_fig("05_cate_spatial")
    plt.show()
except Exception as e:
    print(f"Causal Forest no disponible en este entorno: {e}")


In [ ]:
# 4. Sensitivity analysis: Oster (2019) δ para confounding no observado
from stats_helpers import oster_sensitivity

# Modelo controlado (con lat/long)
ols_ctrl = smf.ols('MedHouseVal ~ MedInc + Latitude + Longitude + HouseAge + AveRooms + AveBedrms + Population + AveOccup', df).fit()
beta_ctrl = ols_ctrl.params['MedInc']
r2_ctrl = ols_ctrl.rsquared

# Modelo no controlado (solo MedInc)
ols_unctrl = smf.ols('MedHouseVal ~ MedInc', df).fit()
beta_unctrl = ols_unctrl.params['MedInc']
r2_unctrl = ols_unctrl.rsquared

print(f"β controlado (con lat/long): {beta_ctrl:.4f}, R² = {r2_ctrl:.4f}")
print(f"β no controlado: {beta_unctrl:.4f}, R² = {r2_unctrl:.4f}")

# Oster δ (assuming R²_max = 1.3 * R²_controlado)
r2_max = min(1.3 * r2_ctrl, 1.0)
delta = oster_sensitivity(beta_ctrl, beta_unctrl, r2_ctrl, r2_unctrl, r2_max)
print(f"\nOster δ (R²_max={r2_max:.3f}): {delta:.3f}")
print(f"Interpretación: confounding no observado debería ser {delta:.1f}x más fuerte que confounding observado (lat/long) para explicar el efecto.")

# Sensitivity curve
r2_max_range = np.linspace(r2_ctrl, min(1.3*r2_ctrl, 1.0), 20)
deltas = [oster_sensitivity(beta_ctrl, beta_unctrl, r2_ctrl, r2_unctrl, r) for r in r2_max_range]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(r2_max_range, deltas, 'o-')
ax.axhline(1, color='red', linestyle='--', label='δ=1 (confounding igual)')
ax.set_xlabel('R² máximo (R²_max)')
ax.set_ylabel('δ de Oster')
ax.set_title('Sensitivity Analysis: Oster (2019)')
ax.legend()
save_fig("05_oster_sensitivity")
plt.show()

In [ ]:
# Validación: Spatial Block CV (o fallback documentado)
from stats_helpers import spatial_block_cv
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
import numpy as np

coords = df[['Latitude', 'Longitude']].values
cv_mode = 'spatial_block'
try:
    splits = spatial_block_cv(coords, n_splits=5)
except Exception as e:
    print(f"Spatial block CV no disponible ({e}); usando KFold como fallback.")
    from sklearn.model_selection import KFold
    cv_mode = 'kfold_fallback'
    splits = list(KFold(n_splits=5, shuffle=True, random_state=42).split(coords))

feature_cols = [c for c in df.columns if c not in ['MedHouseVal', 'lat_q', 'resid_ols']]
X_all = df[feature_cols].apply(pd.to_numeric, errors='coerce').values
y_all = df['MedHouseVal'].values

cv_results = {'OLS': [], 'GBT': []}
for fold, (train_idx, test_idx) in enumerate(splits):
    X_tr, X_te = X_all[train_idx], X_all[test_idx]
    y_tr, y_te = y_all[train_idx], y_all[test_idx]

    ols_cv = LinearRegression().fit(X_tr, y_tr)
    pred_ols = ols_cv.predict(X_te)
    cv_results['OLS'].append({'rmse': float(np.sqrt(mean_squared_error(y_te, pred_ols))), 'r2': float(r2_score(y_te, pred_ols))})

    gbr_cv = HistGradientBoostingRegressor(random_state=42, max_iter=100)
    gbr_cv.fit(X_tr, y_tr)
    pred_gbr = gbr_cv.predict(X_te)
    cv_results['GBT'].append({'rmse': float(np.sqrt(mean_squared_error(y_te, pred_gbr))), 'r2': float(r2_score(y_te, pred_gbr))})

cv_summary = {}
for model_name, results in cv_results.items():
    rmse_mean = np.mean([r['rmse'] for r in results])
    r2_mean = np.mean([r['r2'] for r in results])
    cv_summary[model_name] = {'rmse': float(rmse_mean), 'r2': float(r2_mean)}
    print(f"{model_name} CV ({cv_mode}): RMSE={rmse_mean:.4f}, R²={r2_mean:.4f}")


> **Resumen técnico (trazabilidad y alcance):** El análisis parte de datos observacionales georreferenciados (1990). Se comparan especificaciones OLS (bruta y con controles), se inspeccionan residuos y heterogeneidad espacial descriptiva, y se incorporan —solo cuando el entorno lo permite— diagnósticos/modelos espaciales o de heterogeneidad condicional (Moran's I, SAR, Causal Forest, SHAP) además de validación fuera de muestra. La sensibilidad tipo Oster se usa como ejercicio de robustez ante confounders no observados, no como prueba de identificación causal. Toda afirmación numérica en la conclusión debe corresponder a resultados efectivamente calculados en esta ejecución.

## 5. Lo que el censo de 1990 no puede decirnos (¼ cuartilla)

## 6. Cierre argumental y trazabilidad (dato → método → hallazgo → límite)

**Desarrollo argumental (sustantivo):** La historia se fundamenta en la estructura del dataset: variables socioeconómicas y de vivienda medidas en unidades espaciales (*block groups*) con coordenadas. Esa estructura implica que (1) la asociación ingreso–precio puede estar confundida por localización y amenities; (2) la dependencia espacial puede afectar inferencia y errores; (3) la relación puede ser no lineal y heterogénea entre submercados. Por ello, el análisis avanza por capas: descripción y asociación bruta, condicionamiento por observables, diagnóstico espacial, modelos flexibles/predictivos y (cuando están disponibles) extensiones espaciales o de heterogeneidad, siempre contrastando con validación y con una lectura explícita de límites (MAUP, antigüedad de datos, confounders no observados).

**Trazabilidad:** cada bloque de resultados alimenta el siguiente: el dato define el modelo pertinente; el modelo produce estimaciones/diagnósticos; la narrativa solo debe reclamar lo que esas salidas muestran. Donde una herramienta no está disponible en el entorno núcleo, la conclusión se restringe a la evidencia efectivamente obtenida.

### Nota de coherencia narrativa y extensión
El arquetipo de intersección multivariable/confounding espacial es adecuado porque la covariación entre ingreso, ubicación y precio es el núcleo del problema en este dataset. La historia es con datos en la medida en que su argumento depende de esa covariación observada y de las comparaciones cuantitativas realizadas sobre el archivo analizado.

In [ ]:
# Conclusión estructurada (dinámica, solo con resultados disponibles)
from narrative import conclusion_tecnica
from IPython.display import display
import numpy as np

# Coeficientes OLS
b_simple = float(slope) if 'slope' in globals() else np.nan
try:
    b_ctrl = float(ols.params['MedInc'])
    ci_ctrl = ols.conf_int().loc['MedInc'].values
    b_ctrl_ci = (float(ci_ctrl[0]), float(ci_ctrl[1]))
except Exception:
    b_ctrl = np.nan; b_ctrl_ci = (np.nan, np.nan)

# Oster
delta_oster = float(delta) if ('delta' in globals() and np.isfinite(delta)) else np.nan
r2max_txt = f"{r2_max:.3f}" if 'r2_max' in globals() else "N/D"

# Espacial / causal avanzada (si disponible)
sar_txt = f"β_MedInc(SAR)={sar_beta_medinc:.3f}, ρ={sar_rho:.3f}" if (np.isfinite(sar_beta_medinc) and np.isfinite(sar_rho)) else "SAR no estimado en este entorno"
cate_txt = f"CATE media={cate_mean:.3f} [p5={cate_p5:.3f}, p95={cate_p95:.3f}]" if np.isfinite(cate_mean) else "Causal Forest/CATE no estimado en este entorno"
moran_txt = f"Moran's I={mi.get('I'):.4f} (p={mi.get('p_sim', mi.get('p_norm', np.nan)):.4g})" if ('moran_available' in globals() and moran_available) else "Moran's I no disponible en este entorno"

# CV
cv_txt = "no disponible"
try:
    cv_txt = f"CV ({cv_mode}): OLS R²={cv_summary['OLS']['r2']:.3f}, RMSE={cv_summary['OLS']['rmse']:.3f}; GBT R²={cv_summary['GBT']['r2']:.3f}, RMSE={cv_summary['GBT']['rmse']:.3f}"
except Exception:
    pass

parts = []
if np.isfinite(b_simple) and np.isfinite(b_ctrl):
    parts.append(f"Asociación MedInc–MedHouseVal: β simple={b_simple:.3f} vs β con controles y coordenadas={b_ctrl:.3f} (IC95% [{b_ctrl_ci[0]:.3f}, {b_ctrl_ci[1]:.3f}]) en unidades $100k por $10k de ingreso.")
else:
    parts.append("No fue posible reportar de forma completa los coeficientes OLS de referencia en esta ejecución.")
parts.append(f"Diagnóstico espacial: {moran_txt}.")
parts.append(f"Modelos/extensiones adicionales: {sar_txt}; {cate_txt}.")
if np.isfinite(delta_oster):
    parts.append(f"Sensibilidad Oster (exploratoria): δ={delta_oster:.3f} con R²_max={r2max_txt} (no implica identificación causal).")
else:
    parts.append("Sensibilidad Oster no disponible o no interpretable en esta ejecución.")
parts.append(f"Desempeño fuera de muestra: {cv_txt}.")

respuesta = " ".join(parts) + " Lectura: evidencia observacional/condicional sujeta a confounding no observado, MAUP y antigüedad de los datos (1990); no se establece causalidad definitiva."

display(conclusion_tecnica(
    respuesta=respuesta,
    siguiente_paso="Para fortalecer inferencia causal: usar datos más recientes y, si es posible, estrategias cuasi-experimentales (p. ej. discontinuidades o instrumentos plausibles) además de modelos espaciales explícitos; reportar siempre diagnósticos y supuestos junto a las estimaciones.",
    reproducibilidad="Reproducible en entorno núcleo (venv+pip, `requirements.txt`, `build_all.py`) para los componentes base (OLS, PDP, CV, Oster). Componentes espaciales/causales avanzados (SAR, Moran con libpysal/esda, Causal Forest/econml, SHAP) requieren dependencias opcionales y deben reportarse solo cuando se ejecuten."
))
